# MGMT298D: Science and Strategy of AI
## Week 4: Reinforcement Learning & Dynamic Pricing
### UCLA Anderson School of Management

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

np.random.seed(42)
sns.set_style('whitegrid')

## Pricing Environment Setup

In [ ]:
# Define price options and true conversion rates for each price point
PRICES = [5, 10, 15, 20, 25]
TRUE_CONVERSION_RATES = [0.50, 0.35, 0.22, 0.12, 0.05]

# Display price options with expected revenue
price_data = pd.DataFrame({
    'Price': PRICES,
    'Conversion Rate': TRUE_CONVERSION_RATES,
    'Expected Revenue': [p * cr for p, cr in zip(PRICES, TRUE_CONVERSION_RATES)]
})

print("Price Options and Expected Revenue:")
print(price_data.to_string(index=False))
print(f"\\nOptimal arm (highest expected revenue): Price ${PRICES[np.argmax([p * cr for p, cr in zip(PRICES, TRUE_CONVERSION_RATES)])]}")

## Multi-Armed Bandit Agent

In [ ]:
class MultiArmedBandit:
    """Multi-armed bandit agent for dynamic pricing"""
    def __init__(self, num_arms):
        self.num_arms = num_arms
        self.counts = np.zeros(num_arms)  # Number of times each arm was selected
        self.values = np.zeros(num_arms)  # Estimated value of each arm
        self.cumulative_reward = 0
        self.rewards_history = []
    
    def update(self, arm, reward):
        """Update arm estimates with new reward observation"""
        self.counts[arm] += 1
        old_value = self.values[arm]
        self.values[arm] = old_value + (reward - old_value) / self.counts[arm]
        self.cumulative_reward += reward
        self.rewards_history.append(self.cumulative_reward)

def simulate_purchase(arm):
    """Simulate a purchase at given price, return revenue if converted"""
    price = PRICES[arm]
    conversion_rate = TRUE_CONVERSION_RATES[arm]
    converted = np.random.rand() < conversion_rate
    return price if converted else 0

## Strategy 1: Random Arm Selection

In [ ]:
# Random strategy: uniformly select price at each customer
agent_random = MultiArmedBandit(len(PRICES))
num_customers = 1000

for _ in range(num_customers):
    arm = np.random.randint(len(PRICES))
    reward = simulate_purchase(arm)
    agent_random.update(arm, reward)

print(f"Random Strategy Results (n={num_customers}):")
print(f"Total Revenue: ${agent_random.cumulative_reward:.2f}")
print(f"Average Revenue per Customer: ${agent_random.cumulative_reward / num_customers:.2f}")

## Strategy 2: Epsilon-Greedy

In [ ]:
# Epsilon-greedy strategy: balance exploration and exploitation
agent_eg = MultiArmedBandit(len(PRICES))
epsilon = 0.1
num_customers = 1000

for _ in range(num_customers):
    # Explore with probability epsilon, else exploit best arm
    if np.random.rand() < epsilon:
        arm = np.random.randint(len(PRICES))
    else:
        arm = np.argmax(agent_eg.values)
    reward = simulate_purchase(arm)
    agent_eg.update(arm, reward)

print(f"Epsilon-Greedy Strategy Results (epsilon={epsilon}, n={num_customers}):")
print(f"Total Revenue: ${agent_eg.cumulative_reward:.2f}")
print(f"Average Revenue per Customer: ${agent_eg.cumulative_reward / num_customers:.2f}")
print("\\nLearned Arm Values:")
learned_values = pd.DataFrame({
    'Price': PRICES,
    'Learned Value': agent_eg.values,
    'Times Selected': agent_eg.counts.astype(int)
})
print(learned_values.to_string(index=False))

## Strategy 3: Upper Confidence Bound (UCB)

In [ ]:
# UCB strategy: select arm with highest upper confidence bound
class UCBAgent(MultiArmedBandit):
    def __init__(self, num_arms, confidence=2.0):
        super().__init__(num_arms)
        self.confidence = confidence
    
    def select_arm(self):
        """Select arm with highest UCB value"""
        ucb_values = np.zeros(self.num_arms)
        for arm in range(self.num_arms):
            if self.counts[arm] == 0:
                ucb_values[arm] = float('inf')
            else:
                exploration_bonus = self.confidence * np.sqrt(np.log(sum(self.counts)) / self.counts[arm])
                ucb_values[arm] = self.values[arm] + exploration_bonus
        return np.argmax(ucb_values)

agent_ucb = UCBAgent(len(PRICES), confidence=2.0)
num_customers = 1000

for _ in range(num_customers):
    arm = agent_ucb.select_arm()
    reward = simulate_purchase(arm)
    agent_ucb.update(arm, reward)

print(f"UCB Strategy Results (confidence=2.0, n={num_customers}):")
print(f"Total Revenue: ${agent_ucb.cumulative_reward:.2f}")
print(f"Average Revenue per Customer: ${agent_ucb.cumulative_reward / num_customers:.2f}")
print("\\nLearned Arm Values:")
learned_values_ucb = pd.DataFrame({
    'Price': PRICES,
    'Learned Value': agent_ucb.values,
    'Times Selected': agent_ucb.counts.astype(int)
})
print(learned_values_ucb.to_string(index=False))

## Strategy Comparison

In [ ]:
# Plot cumulative rewards for all three strategies
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(agent_random.rewards_history, label='Random', alpha=0.7, linewidth=2)
ax.plot(agent_eg.rewards_history, label=f'Epsilon-Greedy (ε={epsilon})', alpha=0.7, linewidth=2)
ax.plot(agent_ucb.rewards_history, label='UCB', alpha=0.7, linewidth=2)
ax.set_xlabel('Customer Number', fontsize=12)
ax.set_ylabel('Cumulative Revenue ($)', fontsize=12)
ax.set_title('Cumulative Revenue Over Time: Bandit Strategies', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
summary_df = pd.DataFrame({
    'Strategy': ['Random', 'Epsilon-Greedy', 'UCB'],
    'Total Revenue': [agent_random.cumulative_reward, agent_eg.cumulative_reward, agent_ucb.cumulative_reward],
    'Avg per Customer': [agent_random.cumulative_reward / num_customers, 
                         agent_eg.cumulative_reward / num_customers,
                         agent_ucb.cumulative_reward / num_customers]
})
print("\\nStrategy Summary:")
print(summary_df.to_string(index=False))

## Arm Selection Distribution by Strategy

In [ ]:
# Compare how often each price was selected
x = np.arange(len(PRICES))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, agent_random.counts, width, label='Random', alpha=0.8)
ax.bar(x, agent_eg.counts, width, label=f'Epsilon-Greedy', alpha=0.8)
ax.bar(x + width, agent_ucb.counts, width, label='UCB', alpha=0.8)

ax.set_xlabel('Price ($)', fontsize=12)
ax.set_ylabel('Selection Count', fontsize=12)
ax.set_title('How Often Each Price Was Selected by Strategy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'${p}' for p in PRICES])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Q-Learning Environment

In [ ]:
# Define environment states: (inventory level) x (time of day)
# States: 0=High Inventory, Early; 1=High Inventory, Late; 2=Low Inventory, Early; 3=Low Inventory, Late
# Actions (arms): Same 5 prices as before

class PricingEnvironment:
    """State-dependent pricing environment for Q-learning"""
    def __init__(self):
        self.state = 0
        self.conversion_rate_modifiers = {
            0: 1.0,    # High inventory, early: baseline
            1: 0.8,    # High inventory, late: fewer customers
            2: 1.3,    # Low inventory, early: higher conversion due to scarcity
            3: 1.1     # Low inventory, late: moderate boost
        }
    
    def get_conversion_rate(self, price_idx):
        """Get state-dependent conversion rate"""
        base_rate = TRUE_CONVERSION_RATES[price_idx]
        modifier = self.conversion_rate_modifiers[self.state]
        return min(base_rate * modifier, 1.0)  # Cap at 100%
    
    def step(self, action):
        """Execute action and transition to next state"""
        price = PRICES[action]
        conv_rate = self.get_conversion_rate(action)
        reward = price if np.random.rand() < conv_rate else 0
        
        # Transition to next state (cycle through states)
        self.state = (self.state + 1) % 4
        return reward, self.state
    
    def reset(self):
        """Reset environment to initial state"""
        self.state = np.random.randint(4)
        return self.state

print("State Space: 4 states (inventory x time)")
print("State 0: High Inventory, Early")
print("State 1: High Inventory, Late")
print("State 2: Low Inventory, Early")
print("State 3: Low Inventory, Late")
print("\\nAction Space: 5 prices (5, 10, 15, 20, 25)")

## Q-Learning Agent

In [ ]:
class QLearningAgent:
    """Q-learning agent for dynamic pricing"""
    def __init__(self, num_states, num_actions, learning_rate=0.1, discount_factor=0.95):
        self.num_states = num_states
        self.num_actions = num_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.q_table = np.zeros((num_states, num_actions))
        self.epsilon = 0.1
    
    def choose_action(self, state):
        """Epsilon-greedy action selection"""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            return np.argmax(self.q_table[state])
    
    def update(self, state, action, reward, next_state):
        """Q-learning TD update"""
        current_q = self.q_table[state, action]
        max_next_q = np.max(self.q_table[next_state])
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * max_next_q - current_q)
        self.q_table[state, action] = new_q
    
    def get_policy(self):
        """Extract greedy policy from Q-table"""
        return np.argmax(self.q_table, axis=1)

print("Q-Learning Agent initialized")
print(f"Learning rate: 0.1, Discount factor: 0.95, Exploration rate: 0.1")

## Train Q-Learning Agent

In [ ]:
# Train Q-learning agent for 1000 episodes
env = PricingEnvironment()
agent = QLearningAgent(num_states=4, num_actions=len(PRICES))

num_episodes = 1000
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    episode_reward = 0
    
    for step in range(50):  # 50 steps per episode
        action = agent.choose_action(state)
        reward, next_state = env.step(action)
        agent.update(state, action, reward, next_state)
        episode_reward += reward
        state = next_state
    
    episode_rewards.append(episode_reward)

# Calculate rolling average
window = 50
rolling_avg = pd.Series(episode_rewards).rolling(window=window).mean()

print(f"Training complete: {num_episodes} episodes")
print(f"Mean episode reward: ${np.mean(episode_rewards):.2f}")
print(f"Final 100 episode average: ${np.mean(episode_rewards[-100:]):.2f}")

## Q-Learning Convergence

In [ ]:
# Plot learning curve with rolling average
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(episode_rewards, alpha=0.3, color='steelblue', label='Episode Reward')
ax.plot(rolling_avg, color='darkblue', linewidth=2.5, label=f'Rolling Average (window={window})')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward ($)', fontsize=12)
ax.set_title('Q-Learning Training Progress', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Learned Q-Table and Policy

In [ ]:
# Display Q-table
state_names = ['High Inv, Early', 'High Inv, Late', 'Low Inv, Early', 'Low Inv, Late']
q_table_df = pd.DataFrame(
    agent.q_table,
    index=state_names,
    columns=[f'${p}' for p in PRICES]
)

print("Q-Table (State-Action Values):")
print(q_table_df.round(3))

# Extract and display learned policy
policy = agent.get_policy()
policy_df = pd.DataFrame({
    'State': state_names,
    'Optimal Price': [PRICES[p] for p in policy],
    'Q-Value': [agent.q_table[i, policy[i]] for i in range(len(policy))]
})

print("\\nLearned Pricing Policy:")
print(policy_df.to_string(index=False))

In [ ]:
# Heatmap visualization of Q-table
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(agent.q_table, annot=True, fmt='.2f', cmap='YlGnBu', 
            xticklabels=[f'${p}' for p in PRICES],
            yticklabels=state_names,
            cbar_kws={'label': 'Q-Value'},
            ax=ax, linewidths=0.5)
ax.set_xlabel('Price (Action)', fontsize=12)
ax.set_ylabel('State', fontsize=12)
ax.set_title('Q-Learning Q-Table Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary and Key Insights

### Bandit Strategies
- **Random**: No learning, establishes baseline performance
- **Epsilon-Greedy**: Balances exploration and exploitation, improves over time
- **UCB**: Theoretically optimal exploration strategy, typically outperforms epsilon-greedy

### Q-Learning
- State-aware approach that learns context-dependent optimal pricing
- Converges to stable policy that adapts to different market conditions
- Q-table heatmap reveals which prices are optimal in each state

### Practical Applications
1. **Dynamic pricing platforms** can adapt prices based on inventory and time
2. **Revenue management** systems use similar RL approaches
3. **Recommendation systems** employ multi-armed bandit algorithms
4. **A/B testing** frameworks benefit from bandit approaches over traditional fixed-horizon tests